# Mage-Flow-Turbo — PyTorch BF16 on Kaggle Tesla T4 ×2

## Public bilingual production demo / Demo production công khai song ngữ

This notebook runs **two independent lanes** against the same frozen authority:

- **Lane A — canonical qualification**: one logical text-to-image (T2I)
  trajectory across two Tesla T4 GPUs, exactly `512×512`, `seed 42`, `4`
  denoising steps, `CFG 1.0`, SDPA, BF16 dtype/materialization. This is the
  acceptance authority and it **never becomes variable**.
- **Lane B — production showcase**: a separate, sustained multi-image demo that
  performs its **own single model load** and keeps the instance hot across a
  deterministic workload of many images at `512 / 768 / 1024`, recording
  timing, GPU memory (allocated + reserved), routing and output evidence per
  image, plus a labeled contact-sheet gallery.

Notebook này chạy **hai lane độc lập** dựa trên cùng một authority đã đóng băng:

- **Lane A — kiểm định chuẩn (qualification)**: một quỹ đạo text-to-image (T2I)
  logic duy nhất chạy trên hai GPU Tesla T4, đúng `512×512`, `seed 42`,
  `4` bước denoising, `CFG 1.0`, SDPA, materialization BF16. Đây là authority
  chấp nhận và **không bao giờ bị thay đổi**.
- **Lane B — showcase production**: một demo đa-ảnh bền vững, tách biệt, tự
  thực hiện **một lần nạp model riêng**, giữ model nóng xuyên suốt workload
  xác định nhiều ảnh ở `512 / 768 / 1024`, ghi lại evidence timing, bộ nhớ GPU
  (allocated + reserved), routing và output cho từng ảnh, cùng gallery dạng
  contact sheet có nhãn.

## Before you run / Trước khi chạy

- **Accelerator**: GPU **T4 ×2** (exactly two Tesla T4). / **Tăng tốc**:
  GPU **T4 ×2** (đúng hai Tesla T4).
- **Internet**: **ON**. The notebook bootstraps public source from Git; a
  private ZIP or identity sidecar is never required.
  / Internet **BẬT**: notebook lấy source công khai từ Git; không bao giờ cần
  ZIP nội bộ hay sidecar định danh.
- **Kaggle Model attachment**: `dangkhoa2016/mage-flow-community-mage-flow-turbo`
  mounted read-only at `/kaggle/input/models/dangkhoa2016/
  mage-flow-community-mage-flow-turbo/pytorch/default/1`.
- **Run All exactly once** for a publication-quality run.
  / **Run All đúng một lần** để có bản chạy chất lượng xuất bản.

## Architecture / Kiến trúc

```text
Prompt
  |
Text encoder ------------------------ cuda:0
  |
Transformer block 0 ---------------- cuda:0
  |
activation transfer 0 -> 1
  |
Transformer blocks 1..11 ----------- cuda:1
norm_out / proj_out ----------------- cuda:1
  |
transformer result 1 -> 0
  |
latent / scheduler path ------------- cuda:0
  |
VAE input 0 -> 1
  |
VAE --------------------------------- cuda:1
  |
512×512 RGB image (Lane A) / multi-resolution outputs (Lane B)
```

The transformer sequence `[0..11]` runs **four times** (once per denoising
invocation) in Lane A, and the same explicit dual-T4 placement is applied
inside Lane B while the model stays hot across all showcase images.

/ Dãy transformer `[0..11]` chạy **bốn lần** (mỗi lần một invocation
denoising) trong Lane A, và cùng placement T4 kép được áp dụng trong Lane B
khi model giữ nóng xuyên suốt mọi ảnh showcase.

## Step 1 — Clone the pinned public source / Clone source public đã pin

`SOURCE_REF` may be a **release tag** or an **exact 40-hex commit SHA**; the
bootstrap resolves it, verifies the remote origin, checks out the detached
resolved commit and verifies `HEAD`. The committed default stays
`SOURCE_REF = "v1.0.0"`.

`SOURCE_REF` có thể là **release tag** hoặc **SHA 40 ký tự chính xác**;
bootstrap phân giải nó, xác minh remote origin, checkout detached commit đã
phân giải và xác minh `HEAD`. Giá trị mặc định được commit giữ nguyên là
`SOURCE_REF = "v1.0.0"`.

In [ ]:
# --- Public Git source bootstrap (tag OR exact SHA) ---
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU.git"
SOURCE_REF = os.environ.get("SOURCE_REF", "v1.0.0")  # release tag or 40-hex SHA

def run(*args, **kwargs):
    result = subprocess.run(args[0], *args[1:], **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {args[0]}")
    return result

REPO = WORK / "Mage-Flow-Turbo-PyTorch-BF16-on-T4x2-GPU"
if not REPO.exists():
    run(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
run(["git", "-C", str(REPO), "fetch", "--tags", "--force"])
run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_REF])
origin = run(["git", "-C", str(REPO), "remote", "get-url", "origin"],
             capture_output=True, text=True).stdout.strip()
assert origin == REPO_URL, f"unexpected origin: {origin}"
head = run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
           capture_output=True, text=True).stdout.strip()
assert len(head) == 40 and all(c in "0123456789abcdef" for c in head), head
print("SOURCE_REF=", SOURCE_REF)
print("RESOLVED_HEAD=", head)
print("ORIGIN_VERIFIED=", origin)
sys.path.insert(0, str(REPO))
REPO_ROOT = REPO
os.chdir(REPO_ROOT)

## Step 2 — Environment and GPU inventory / Môi trường và kiểm kê GPU

Print the frozen environment and confirm the live inventory is exactly two
Tesla T4 GPUs before any model load.

/ In environment đã đóng băng và xác nhận inventory là đúng hai GPU Tesla T4
trước khi nạp bất kỳ model nào.

In [ ]:
# --- Environment and GPU inventory ---
import json
from mage_t4x2.environment import environment_summary
from scripts import gpu_session as gs

print(json.dumps(environment_summary(), indent=2)[:3000])
pre = gs.preflight_gpu(require_t4x2=False)
print("T4x2_OK=", pre.get("t4x2_ok"))
print("DEVICE_COUNT=", pre.get("device_count"))
for gpu in pre.get("inventory", [])[:2]:
    print(gpu.get("name"), gpu.get("compute_capability"), gpu.get("total_memory_gib"))

## Step 3 — qualified runtime integrity / Tính toàn vẹn runtime qualified

Verify the frozen qualified runtime baseline and the canonical public
contract before live inference. Failure here stops the notebook.

/ Xác minh runtime baseline đã qualified và canonical public contract trước
khi chạy live. Lỗi tại đây sẽ dừng notebook.

In [ ]:
# --- Source / runtime integrity ---
from mage_t4x2.runtime_baseline import verify_runtime_baseline
from public_demo.contract import CANONICAL, CANONICAL_PROMPT

baseline = verify_runtime_baseline()
print("RUNTIME_BASELINE=", baseline.get("status"))
print("ENTRIES=", len(baseline.get("entries", [])))
assert baseline.get("status") == "PASS"
CANONICAL.validate()
print("CANONICAL_CONTRACT=VALID")
print("prompt=", CANONICAL_PROMPT)
print("seed=", CANONICAL.seed, "steps=", CANONICAL.steps, "cfg=", CANONICAL.cfg_scale)
print("blocks=", CANONICAL.num_transformer_blocks, "split=", CANONICAL.split_block)

## Step 4 — Canonical model / source resolution / Phân giải model chuẩn

Resolve the owner-qualified Kaggle model mount for the canonical contract.
Remote fallback is forbidden for both lanes.

/ Phân giải mount model Kaggle đủ điều kiện của chủ sở hữu cho canonical
contract. Fallback từ xa bị cấm ở cả hai lane.

In [ ]:
# --- Canonical model resolution ---
from mage_t4x2.model_provenance import ModelPathResolver
from public_demo.contract import CANONICAL, require_public_demo_model_path

resolver = ModelPathResolver(
    candidates=[str(CANONICAL.model_path)],
    required_rel_files=[
        "model_index.json",
        "transformer/config.json",
        "transformer/diffusion_pytorch_model.safetensors",
        "scheduler/scheduler_config.json",
    ],
)
resolution = resolver.resolve(fallback_id=None)
require_public_demo_model_path(resolution["selected_path"])
print("MODEL_SOURCE=", resolution.get("model_source"))
print("SELECTED=", resolution["selected_path"])
print("REQUIRED_FILES_PRESENT=", resolution.get("required_files_present"))

## Step 5 — Lane A: canonical qualification / Kiểm định chuẩn

The canonical public runner performs its own single model load and one
canonical T2I trajectory. It accepts **no** prompt/seed/resolution overrides
and fails closed on any contract mismatch.

/ Runner public chuẩn tự thực hiện một lần nạp model và một quỹ đạo T2I
canonical. Runner **không** nhận bất kỳ ghi đè prompt/seed/độ phân giải nào và
fail-closed nếu lệch contract.

In [ ]:
# --- Lane A: run the canonical qualification ---
from public_demo.runner import run_public_demo

lane_a = run_public_demo(project_root=str(REPO_ROOT))
print("LANE_A_STATUS=", lane_a["status"])
print("LANE_A_RUN_ID=", lane_a["run_id"])
print("LANE_A_SUMMARY=", lane_a["summary_path"])
if lane_a["status"] != "PASS":
    print(lane_a.get("verdict_lines"))
    raise SystemExit("Lane A qualification must PASS before showing any showcase.")

## Step 6 — Canonical live evidence summary / Tóm tắt evidence live

A concise view of the Lane A gates and accepted facts.

/ Tóm tắt ngắn gọn các gate Lane A và các sự kiện được chấp nhận.

In [ ]:
# --- Concise Lane A result and verdict lines ---
print("LANE_A_VERDICT_LINES:")
for line in lane_a["verdict_lines"]:
    print("  ", line)
print()
print("LANE_A_FINAL_VERDICT=", [l for l in lane_a["verdict_lines"]
                                if l.startswith("PUBLIC_DEMO_FINAL_VERDICT")] or "n/a")

## Step 7 — Canonical device placement / Phân bố thiết bị

Show transformer block ownership across the two T4 GPUs.

/ Trình bày quyền sở hữu block transformer trên hai GPU T4.

In [ ]:
# --- Device placement table (from persisted Lane-A summary) ---
import json

with open(lane_a["summary_path"], encoding="utf-8") as handle:
    lane_a_summary = json.load(handle)

plan = lane_a_summary.get("placement") or {}

if lane_a["status"] == "PASS":
    assert plan, "Lane A PASS but persisted placement evidence is empty"

print("text_encoder=", plan.get("text_encoder"))
print("vae=", plan.get("vae"))
print("transformer_blocks_0..11:")
for key, device in plan.items():
    if str(key).startswith("block "):
        print(f"  {key} = {device}")
print("PLACEMENT_KEYS=", sorted(plan))


## Step 8 — Multistep routing evidence / Evidence routing đa bước

Lane A routing reduced from captured telemetry: invoked blocks, boundary
transfers and returns across the four denoising steps.

/ Routing Lane A rút ra từ telemetry: các block được gọi, transfer biên và
return qua bốn bước denoising.

In [ ]:
# --- Multistep routing summary (Lane A, persisted summary) ---
routing = lane_a_summary.get("routing") or {}

if lane_a["status"] == "PASS":
    assert routing, "Lane A PASS but persisted routing evidence is empty"

print("observed_invocations=", routing.get("observed_transformer_invocations"))
print("transfer_0_to_1_count=", routing.get("transfer_0_to_1_count"))
print("transformer_return_1_to_0_count=", routing.get("transformer_return_1_to_0_count"))
print("vae_input_transfer_0_to_1_count=", routing.get("vae_input_transfer_0_to_1_count"))
print("gpu0_participation=", routing.get("gpu0_participation"))
print("gpu1_participation=", routing.get("gpu1_participation"))
print("single_t2i_instance=", routing.get("single_t2i_instance"))
print("block_order_valid=", routing.get("block_order_valid"))


## Step 9 — Lane B: production showcase / Khởi động showcase

Lane B is a **separate** sustained multi-image demo with its **own single
model load** (never sharing Lane A). It applies the same explicit dual-T4
placement, keeps the model hot, and executes the deterministic showcase table
(`s01..s24`, resolutions `512 / 768 / 1024`, seeds `1001..1024`, 4 steps,
CFG 1.0). Evidence lands under `artifacts/public-showcase/<run_id>/`.

/ Lane B là demo đa-ảnh bền vững **riêng biệt** với **một lần nạp model của
chính nó** (không bao giờ dùng chung với Lane A). Nó áp dụng cùng placement
T4 kép, giữ model nóng, và chạy bảng showcase xác định (`s01..s24`, độ phân
giải `512 / 768 / 1024`, seeds `1001..1024`, 4 bước, CFG 1.0). Evidence lưu
tại `artifacts/public-showcase/<run_id>/`.

In [ ]:
# --- Lane B: run the production showcase ---
from public_demo.showcase import run_public_showcase

lane_b = run_public_showcase(project_root=str(REPO_ROOT))
print("LANE_B_STATUS=", lane_b["status"])
print("LANE_B_RUN_ID=", lane_b["run_id"])
print("LANE_B_SUMMARY=", lane_b["summary_path"])
for line in lane_b["verdict_lines"]:
    print("  ", line)
if lane_b["status"] != "PASS":
    raise SystemExit("Lane B showcase must PASS.")

## Step 10 — Per-image performance table / Bảng hiệu năng từng ảnh

Every executed showcase case is listed with resolution, seed, wall time, GPU
allocated/reserved peaks on both T4 GPUs, output SHA-256 and status.

/ Mỗi case showcase đã chạy được liệt kê với độ phân giải, seed, thời gian,
đỉnh allocated/reserved trên cả hai GPU T4, SHA-256 output và trạng thái.

In [ ]:
# --- Per-image table rendered from evidence ---
import json
summary = json.load(open(lane_b["summary_path"]))
rows = summary.get("case_results", [])
print(f"{'Case':<5} {'Category':<22} {'Res':<7} {'Seed':<6} {'Time(s)':<9} "
      f"{'GPU0 all':<10} {'GPU0 res':<10} {'GPU1 all':<10} {'GPU1 res':<10} "
      f"{'SHA256':<12} Status")
for r in rows:
    ok = (r.get("observation_ok", r.get("validation_ok"))
          and r.get("block_order_valid") and r.get("adapter_detached"))
    print(f"{r.get('case_id'):<5} {str(r.get('category', ''))[:20]:<22} "
          f"{int(r.get('resolution')):<7} "
          f"{int(r.get('seed')):<6} {r.get('elapsed_seconds', 0):<9.3f} "
          f"{r.get('peak_memory_gpu0_bytes', 0):<10} {r.get('peak_memory_reserved_gpu0_bytes', 0):<10} "
          f"{r.get('peak_memory_gpu1_bytes', 0):<10} {r.get('peak_memory_reserved_gpu1_bytes', 0):<10} "
          f"{str(r.get('output_sha256', ''))[:12]:<12} {'PASS' if ok else 'FAIL'}")
print("TOTAL_IMAGES=", summary.get("cases_executed"))
print("SUMMIT_VERDICT=", summary.get(summary.get("verdict_key", "PUBLIC_SHOWCASE_FINAL_VERDICT")))
print("---- Deterministic showcase prompt map ----")
for r in rows:
    print(f"{r.get('case_id')} | {str(r.get('category', '')):<22} | {r.get('prompt', '')}")


## Step 11 — Aggregate resolution and memory / Tổng hợp độ phân giải và bộ nhớ

Show the aggregate showcase summary: model load time, total inference time,
wall time, images per minute, global GPU peaks and per-resolution statistics.

/ Trình bày tổng hợp showcase: thời gian nạp model, tổng thời gian inference,
thời gian wall, ảnh mỗi phút, đỉnh GPU toàn cục và thống kê theo độ phân giải.

In [ ]:
# --- Aggregate showcase summary ---
import json
summary = json.load(open(lane_b["summary_path"]))
agg = summary.get("aggregate", {})
for key in ("showcase_model_load_seconds", "showcase_total_inference_seconds",
            "showcase_wall_seconds", "total_images", "images_per_minute",
            "global_gpu0_peak_allocated", "global_gpu0_peak_reserved",
            "global_gpu1_peak_allocated", "global_gpu1_peak_reserved"):
    print(f"{key}={agg.get(key)}")
print("per_resolution:")
print(json.dumps(agg.get("per_resolution", {}), indent=2))
print("observed_on=", agg.get("observed_on"))

## Step 12 — Individual generated outputs / Các output generated riêng lẻ

Every executed showcase case is an **independent model inference output**:
one PNG per case, saved separately under `outputs/`. Each output below is
shown individually, grouped by resolution, with its case id, category,
exact deterministic prompt, resolution, seed, inference seconds and output
SHA-256. The exact prompt is read from persisted Lane B case evidence and
printed immediately above the image.

/ Mỗi case showcase đã chạy là một **output inference độc lập**: một PNG mỗi
case, lưu riêng trong `outputs/`. Mỗi output dưới đây được hiển thị riêng,
theo độ phân giải, kèm case id, category, prompt xác định chính xác,
độ phân giải, seed, giây inference và SHA-256 output. Prompt chính xác được
đọc từ lane B evidence đã lưu và in ngay phía trên ảnh.

In [ ]:
# --- Individual generated outputs (primary presentation) ---
from pathlib import Path
from IPython.display import Image as IPImage, Markdown, display

summary = json.load(open(lane_b["summary_path"], encoding="utf-8"))
rows = list(summary.get("case_results", []))
run_dir = Path(lane_b["summary_path"]).parent
outputs_dir = run_dir / "outputs"

displayed = 0

for resolution in (512, 768, 1024):
    resolution_rows = [
        row for row in rows
        if int(row.get("resolution")) == resolution
    ]

    if not resolution_rows:
        continue

    display(Markdown(f"### {resolution}×{resolution} outputs"))

    for row in resolution_rows:
        output_path = outputs_dir / str(row["output_rel"])
        prompt = str(row.get("prompt", "")).strip()

        assert output_path.is_file(), output_path
        assert prompt, f"missing prompt for {row['case_id']}"

        display(
            Markdown(
                f"**{row['case_id']} — {row.get('category', '')}**  \n"
                f"**Prompt:** {prompt}  \n"
                f"Resolution: `{resolution}×{resolution}` · "
                f"Seed: `{row['seed']}` · "
                f"Inference: `{float(row['elapsed_seconds']):.3f} s`  \n"
                f"SHA-256: `{row['output_sha256']}`"
            )
        )

        display(IPImage(filename=str(output_path)))
        displayed += 1

print("INDIVIDUAL_OUTPUT_IMAGES=", displayed)
assert displayed == int(summary["cases_executed"]), (
    "every executed case output must be displayed individually"
)


## Step 13 — Derived contact sheet and final combined verdict / Contact sheet dẫn xuất và kết luận cuối

The images above are independent PNG outputs generated by separate inference
cases. The contact sheet below is created afterward only as a visual index.
It is not a direct model output.

Các ảnh phía trên là các PNG độc lập được sinh bởi từng lượt inference riêng.
Contact sheet bên dưới chỉ được tạo sau đó để làm chỉ mục trực quan.
Nó không phải là output trực tiếp của model.

Below it, the final combined verdict prints both lane results so a pretty
gallery can never override a failed canonical qualification.

/ Bên dưới, kết luận cuối cùng in kết quả cả hai lane để một gallery đẹp
không bao giờ che được một canonical qualification thất bại.

In [ ]:
# --- Derived contact sheet and final combined verdict ---
import os
from IPython.display import Image as IPImage, display

summary = json.load(open(lane_b["summary_path"], encoding="utf-8"))

print("CONTACT_SHEET_IS_DERIVED_ARTIFACT=True")
print("CONTACT_SHEET_IS_MODEL_OUTPUT=False")

gallery = summary.get("gallery") or {}
gallery_path = gallery.get("path")
if gallery_path and os.path.isfile(gallery_path):
    display(IPImage(filename=gallery_path))
print("GALLERY_IMAGES=", gallery.get("count"))

lane_a_verdict = [l for l in lane_a["verdict_lines"] if l.startswith("PUBLIC_DEMO_FINAL_VERDICT")]
lane_b_verdict = [l for l in lane_b["verdict_lines"] if l.startswith("PUBLIC_SHOWCASE_FINAL_VERDICT")]
print()
print(" or ".join(lane_a_verdict) or "PUBLIC_DEMO_FINAL_VERDICT=n/a")
print(" or ".join(lane_b_verdict) or "PUBLIC_SHOWCASE_FINAL_VERDICT=n/a")
combined = lane_a["status"] == "PASS" and lane_b["status"] == "PASS"
public_notebook_final_verdict = "PASS" if combined else "FAIL"
print("COMBINED_VERDICT=", public_notebook_final_verdict)
print(f"PUBLIC_NOTEBOOK_FINAL_VERDICT={public_notebook_final_verdict}")
if not combined:
    raise SystemExit(
        "PUBLIC_NOTEBOOK_FINAL_VERDICT=FAIL: "
        "Lane A and Lane B must both PASS."
    )

## Reproducibility and limitations / Tái lập và giới hạn

- Proof rests on **live evidence**: telemetry, dtype audits, load events,
  per-image SHA-256 digests and the evidence manifest written per lane.
  / Bằng chứng dựa trên **evidence live**: telemetry, dtype audits, load
  events, digest SHA-256 từng ảnh và manifest evidence của từng lane.
- CPU/static CI validates contracts but does **not** pretend to replace real
  dual-T4 execution. / CI CPU/static xác minh contract nhưng **không** thay
  thế việc chạy thật trên T4 kép.
- BF16 is **dtype/materialization**, not native BF16 Tensor Core acceleration
  on Tesla T4. / BF16 là **dtype/materialization**, không phải tăng tốc
  Tensor Core BF16 gốc trên Tesla T4.
- Lane B images are deterministic (seed-qualified filenames) but the showcase
  is a performance/output demonstration; the canonical qualification remains
  Lane A. / Ảnh Lane B xác định (tên file gắn seed) nhưng showcase là minh
  họa hiệu năng/output; kiểm định chuẩn vẫn là Lane A.
- The committed `SOURCE_REF = "v1.0.0"` resolves via the Git bootstrap; for
  pre-tag qualification an uncommitted temporary copy pins the exact V4 HEAD
  SHA. / `SOURCE_REF = "v1.0.0"` mặc định được commit phân giải qua Git
  bootstrap; để qualification trước tag, một bản sao tạm không commit pin
  SHA HEAD V4 chính xác.